# HubbardBath 09 — Physical vs algorithmic thermalisation

This notebook combines the two halves of HubbardBath on the **same half-filled thermal target**.

Physical difficulty is measured using the Liouvillian gap and trace-distance mixing time. Algorithmic difficulty is measured using the minimum bounded thermal-polynomial degree at fixed approximation error.

We do not assume in advance that these quantities should correlate.

## 1. Consistency check: same target state

The open-system calculations use the canonical two-electron sector. The block encoding acts on the full four-qubit Fock space. For a fair comparison, the polynomial thermal amplitude is therefore projected into the $N=2$ sector and renormalized before its Gibbs-state error is evaluated.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'HubbardBath']
for c in candidates:
    if (c/'src'/'comparison.py').exists():
        repo_root=c.resolve(); break
else:
    raise FileNotFoundError('Run from the HubbardBath repository')
if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from src.comparison import comparison_case, comparison_summary


## 2. Inspect one reference point

We begin with the same $U/t=4$, $\beta t=1$ reference used throughout earlier milestones.

In [ ]:
reference=comparison_case(U=4.0,beta=1.0)
for key,value in reference.items():
    print(f'{key:38s}: {value}')


## 3. Common 20-point parameter grid

The physical and algorithmic routes are evaluated independently for the same values of interaction and inverse temperature.

In [ ]:
U_values=[0.0,2.0,4.0,6.0,8.0]
beta_values=[0.25,0.5,1.0,2.0]
rows=[]
for U in U_values:
    for beta in beta_values:
        row=comparison_case(U=U,beta=beta)
        rows.append(row)
        print(
            f'U={U:>3.0f} beta={beta:>4.2f} '
            f'tmix={row["mixing_time"]:.4g} '
            f'1/gap={row["inverse_liouvillian_gap"]:.4g} '
            f'degree={row["thermal_polynomial_degree"]:>3d} '
            f'valid={row["comparison_valid"]}'
        )


## 4. Overall correlations

Pearson correlation probes an approximately linear relationship; Spearman correlation probes rank-monotonic agreement. Both are descriptive here.

In [ ]:
summary=comparison_summary(rows)
summary['mixing_time_vs_degree'], summary['inverse_gap_vs_degree']


In [ ]:
valid=[row for row in rows if row['comparison_valid']]
plt.figure(figsize=(7,4.8))
for beta in beta_values:
    group=[row for row in valid if row['beta_t']==beta]
    plt.scatter(
        [row['thermal_polynomial_degree'] for row in group],
        [row['mixing_time'] for row in group],
        label=fr'$\beta t={beta}$'
    )
plt.xlabel('Minimum thermal-polynomial degree')
plt.ylabel(r'Physical mixing time $t_{\rm mix}$')
plt.title('Physical vs algorithmic thermalisation difficulty')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


## 5. Check whether temperature is driving the overall trend

A strong overall correlation can be misleading if both quantities simply increase as temperature falls. We therefore inspect correlations at fixed $\beta$ and at fixed $U$.

In [ ]:
print('Within fixed beta:')
for item in summary['within_beta_mixing_vs_degree']:
    print(item)

print('\nWithin fixed U:')
for item in summary['within_U_mixing_vs_degree']:
    print(item)


## 6. Interpretation discipline

The physical time scale depends on the chosen bath model and rate scale. The algorithmic degree depends on the LCU normalization and polynomial tolerance. Therefore this experiment tests **trend alignment in one controlled finite model**, not a universal equivalence between open-system relaxation and QSVT complexity.

The numerical result itself should be interpreted only after the experiment is run.